# Notebook 3: Student Model + Full System
Knowledge distillation student, dynamic thresholds, uncertainty scoring,
dual-mode pipeline, SHAP with human explanations, scenario simulation, fairness audit, rule overlay.

In [1]:
import pandas as pd
import numpy as np
import json, pickle, warnings, gc
warnings.filterwarnings('ignore')


from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, roc_auc_score, average_precision_score
from scipy.stats import spearmanr
import shap
import matplotlib.pyplot as plt

thick = pd.read_parquet('thick_with_soft_labels.parquet')
with open('feature_sets.json') as f:
    fs = json.load(f)
STUDENT_FEATURES = fs['student']

X        = thick[STUDENT_FEATURES]
y_soft   = thick['SOFT_LABEL']
y_hard   = thick['TARGET']
is_thin  = thick['IS_THIN_FILE']
mode_col = thick['USER_MODE']
sig_col  = thick['SIGNAL_STRENGTH']

print(f'Dataset: {X.shape} | Student features: {len(STUDENT_FEATURES)}')
print(f'Thin-file users: {is_thin.sum()} ({is_thin.mean()*100:.1f}%)')

Dataset: (307511, 30) | Student features: 30
Thin-file users: 2240 (0.7%)


## 1. Train/Val Split

In [2]:
X_tr, X_val, y_tr, y_val, yh_tr, yh_val, thin_tr, thin_val, mode_tr, mode_val, sig_tr, sig_val = \
    train_test_split(
        X, y_soft, y_hard, is_thin, mode_col, sig_col,
        test_size=0.2, stratify=y_hard, random_state=42
    )
print(f'Train: {X_tr.shape} | Val: {X_val.shape}')

Train: (246008, 30) | Val: (61503, 30)


## 2. Dual-Mode Student Models
Mode A (near-thin): has partial signals → model trained on near-thin + thick users
Mode B (pure-thin): completely blank → model trained only on demographic/financial features
This matters because the feature importance structure is different for each population.

In [3]:
# ── Monotonic constraints for student (mirrors teacher design) ────────────
# Financial ratios the student has access to
STUDENT_MONOTONE_MAP = {
    'EMI_BURDEN':          1,   # higher EMI vs income → riskier
    'LOAN_TO_INCOME':      1,   # bigger loan vs income → riskier
    'DEBT_TO_INCOME':      1,
    'ANNUITY_TO_INCOME':   1,
    'FINANCIAL_PRESSURE':  1,   # composite pressure index
    'CREDIT_TO_GOODS':     1,
    'EMPLOYED_YEARS':     -1,   # longer tenure → safer
    'STABILITY_SCORE':    -1,   # higher stability → safer
    'INCOME_ADEQUACY':    -1,   # higher adequacy → safer
    'INCOME_STABILITY':   -1,
    'ASSET_SCORE':        -1,   # assets → safer
    'AGE_YEARS':          -1,
    'INCOME_PER_PERSON':  -1,   # more income per person → safer
    'CHILDREN_RATIO':      1,   # more dependents → riskier
}

def train_student(X_subset, y_subset, label):
    monotone = tuple(STUDENT_MONOTONE_MAP.get(f, 0) for f in STUDENT_FEATURES)
    model = XGBRegressor(
        n_estimators          = 400,
        max_depth             = 5,
        learning_rate         = 0.05,
        subsample             = 0.8,
        colsample_bytree      = 0.7,   # reduced: fewer demographic cols per tree
        colsample_bylevel     = 0.8,   # added: extra regularisation
        reg_alpha             = 0.5,   # increased L1 → sparsity on proxy features
        reg_lambda            = 2.0,   # increased L2
        monotone_constraints  = monotone,
        objective             = 'reg:squarederror',
        eval_metric           = 'rmse',
        early_stopping_rounds = 30,
        random_state          = 42,
        tree_method           = 'hist',
        verbosity             = 0,
    )
    # internal val for early stopping
    Xm_tr, Xm_val, ym_tr, ym_val = train_test_split(
        X_subset, y_subset, test_size=0.15, random_state=0
    )
    model.fit(Xm_tr, ym_tr, eval_set=[(Xm_val, ym_val)], verbose=False)
    print(f'{label}: best_iteration={model.best_iteration}')
    return model

# Mode A: near-thin users + thick users (has some signal)
mask_A = mode_tr.isin(['A_near_thin', 'C_thick'])
student_A = train_student(X_tr[mask_A], y_tr[mask_A], 'Student-A (near-thin)')

# Mode B: pure-thin only (completely no history)
mask_B = mode_tr == 'B_pure_thin'
student_B = train_student(X_tr[mask_B], y_tr[mask_B], 'Student-B (pure-thin)')

print('Both student models trained.')


Student-A (near-thin): best_iteration=399
Student-B (pure-thin): best_iteration=398
Both student models trained.


## 3. Routing & Evaluation

In [4]:
def route_predict(X, mode_series, sig_series):
    """Route each user to the appropriate student model based on signal strength."""
    preds = np.zeros(len(X))
    idx = X.index

    mask_A = mode_series.isin(['A_near_thin', 'C_thick'])
    mask_B = mode_series == 'B_pure_thin'

    if mask_A.sum() > 0:
        preds[mask_A.values] = np.clip(student_A.predict(X[mask_A]), 0, 1)
    if mask_B.sum() > 0:
        preds[mask_B.values] = np.clip(student_B.predict(X[mask_B]), 0, 1)

    return preds

val_preds = route_predict(X_val, mode_val, sig_val)

# Evaluate on thin-file subset
thin_mask  = thin_val.values == 1
thick_mask = thin_val.values == 0

for mask, label in [(thin_mask, 'Thin-file'), (thick_mask, 'Thick-file')]:
    if mask.sum() < 10: continue
    p = val_preds[mask]
    t = yh_val.values[mask]
    if len(np.unique(t)) < 2: continue
    print(f'{label:12s} | n={mask.sum():5d} | ROC-AUC={roc_auc_score(t,p):.4f} | PR-AUC={average_precision_score(t,p):.4f}')

Thin-file    | n=  447 | ROC-AUC=0.7156 | PR-AUC=0.1503
Thick-file   | n=61056 | ROC-AUC=0.6862 | PR-AUC=0.1717


## 4. Dynamic Decision Thresholds
Percentile-based thresholds adapt to population risk distribution.
60th percentile = approval cutoff, 90th = rejection cutoff.
Produces realistic APPROVE/REVIEW/REJECT split (~60/30/10).

In [5]:
# Compute thresholds from training distribution (thin-file users)
train_thin_mask = mode_tr.isin(['A_near_thin', 'B_pure_thin'])
train_thin_preds = route_predict(
    X_tr[train_thin_mask],
    mode_tr[train_thin_mask],
    sig_tr[train_thin_mask]
)

APPROVE_THRESHOLD = np.percentile(train_thin_preds, 60)  # bottom 60% → APPROVE
REJECT_THRESHOLD  = np.percentile(train_thin_preds, 90)  # top 10% → REJECT

print(f'Dynamic thresholds (from training distribution):')
print(f'  APPROVE  : p < {APPROVE_THRESHOLD:.4f}')
print(f'  REVIEW   : {APPROVE_THRESHOLD:.4f} ≤ p < {REJECT_THRESHOLD:.4f}')
print(f'  REJECT   : p ≥ {REJECT_THRESHOLD:.4f}')

# Save thresholds for inference
with open('thresholds.json', 'w') as f:
    json.dump({'approve': APPROVE_THRESHOLD, 'reject': REJECT_THRESHOLD}, f)

# Verify realistic distribution on val thin-file
thin_val_preds = val_preds[thin_mask]
decisions = [
    'APPROVE' if p < APPROVE_THRESHOLD else 'REJECT' if p >= REJECT_THRESHOLD else 'REVIEW'
    for p in thin_val_preds
]
d_series = pd.Series(decisions)
print(f'\nDecision distribution (thin-file val):')
print(d_series.value_counts(normalize=True).round(3))

Dynamic thresholds (from training distribution):
  APPROVE  : p < 0.0825
  REVIEW   : 0.0825 ≤ p < 0.1492
  REJECT   : p ≥ 0.1492

Decision distribution (thin-file val):
APPROVE    0.644
REVIEW     0.273
REJECT     0.083
Name: proportion, dtype: float64


## 5. Hard Business Rules Overlay
No purely ML system in production. Rules handle edge cases the model might miss and increase lender trust.

In [6]:
def apply_business_rules(user_row: dict, ml_decision: str, ml_prob: float):
    """
    Applies hard rules AFTER ML decision. Returns (final_decision, rule_triggered).
    Rules can only make a decision more conservative — never upgrade REJECT to APPROVE.
    """
    emi_burden      = user_row.get('EMI_BURDEN', 0)
    loan_to_income  = user_row.get('LOAN_TO_INCOME', 0)
    income_adequacy = user_row.get('INCOME_ADEQUACY', 10)
    age             = user_row.get('AGE_YEARS', 35)

    # Rule 1: EMI > 60% of income → always REJECT
    if emi_burden > 0.60:
        return 'REJECT', 'EMI burden exceeds 60% of income'

    # Rule 2: Loan > 10x annual income → force REVIEW minimum
    if loan_to_income > 10 and ml_decision == 'APPROVE':
        return 'REVIEW', 'Loan amount exceeds 10x annual income'

    # Rule 3: Very low income adequacy with APPROVE → bump to REVIEW
    if income_adequacy < 0.5 and ml_decision == 'APPROVE':
        return 'REVIEW', 'Income insufficient relative to loan obligations'

    # Rule 4: Applicant under 21 → require REVIEW
    if age < 21 and ml_decision == 'APPROVE':
        return 'REVIEW', 'Applicant age below 21 — manual review required'

    return ml_decision, None

print('Business rules defined: EMI cap, loan ratio cap, income adequacy, age floor.')

Business rules defined: EMI cap, loan ratio cap, income adequacy, age floor.


## 6. Credit Score Mapping
Industry-standard log-odds scaling with increased spread (factor=100 vs typical 75).
Range: 300–900. Higher spread = better separation between risk tiers.

In [7]:
def prob_to_score(p, factor=100, base_score=600):
    """Log-odds scaling. factor=100 gives wider spread than typical 75.
    Accepts scalar or numpy array.
    """
    p = np.clip(p, 1e-6, 1 - 1e-6)
    log_odds = np.log(p / (1 - p))
    score = base_score - factor * log_odds
    result = np.clip(score, 300, 900).astype(int)
    # Return scalar int when given a scalar, array otherwise
    return int(result) if np.ndim(result) == 0 else result

def get_tier(score):
    if score >= 750: return 'Excellent'
    if score >= 700: return 'Good'
    if score >= 650: return 'Fair'
    if score >= 600: return 'Poor'
    return 'Very Poor'

def get_ml_decision(prob, approve_thresh, reject_thresh):
    if prob < approve_thresh:  return 'APPROVE'
    if prob >= reject_thresh:  return 'REJECT'
    return 'REVIEW'

# Verify spread
for p in [0.02, 0.05, 0.10, 0.15, 0.25, 0.40, 0.60]:
    s = prob_to_score(p)
    print(f'p={p:.2f} → score={s:3d} | {get_tier(s)}')


p=0.02 → score=900 | Excellent
p=0.05 → score=894 | Excellent
p=0.10 → score=819 | Excellent
p=0.15 → score=773 | Excellent
p=0.25 → score=709 | Good
p=0.40 → score=640 | Poor
p=0.60 → score=559 | Very Poor


## 7. SHAP Explainability with Human-Readable Reasons
TreeExplainer is exact for tree models. Feature names mapped to plain English sentences.

In [8]:
# Build explainers for both student models
bg_A = shap.maskers.Independent(X_tr[mode_tr.isin(['A_near_thin','C_thick'])], max_samples=500)
bg_B = shap.maskers.Independent(X_tr[mode_tr == 'B_pure_thin'], max_samples=500)

explainer_A = shap.TreeExplainer(student_A, data=bg_A)
explainer_B = shap.TreeExplainer(student_B, data=bg_B)

# Free background masker memory — explainers have already consumed them
del bg_A, bg_B
gc.collect()

# SHAP summary on thin-file val users
X_thin_val    = X_val[thin_val.values == 1]
mode_thin_val = mode_val[thin_val.values == 1]

X_near = X_thin_val[mode_thin_val.isin(['A_near_thin','C_thick'])]
X_pure = X_thin_val[mode_thin_val == 'B_pure_thin']

SHAP_CHUNK = 500  # max rows computed in one go to cap RAM

def _chunked_shap(explainer, X, chunk_size=SHAP_CHUNK):
    """Compute shap_values in chunks to avoid OOM on large subsets."""
    parts = []
    for start in range(0, len(X), chunk_size):
        chunk = X.iloc[start:start + chunk_size]
        parts.append(explainer.shap_values(chunk))
    return np.vstack(parts)

if len(X_near) > 0:
    sv_near = _chunked_shap(explainer_A, X_near)
    shap.summary_plot(sv_near, X_near, feature_names=STUDENT_FEATURES,
                      max_display=15, show=False, title='SHAP — Near-Thin Users')
    plt.tight_layout()
    plt.savefig('shap_near_thin.png', dpi=120, bbox_inches='tight')
    plt.close()           # release figure memory
    del sv_near           # free SHAP matrix (~n_rows × n_features floats)
    gc.collect()

if len(X_pure) > 0:
    sv_pure = _chunked_shap(explainer_B, X_pure)
    shap.summary_plot(sv_pure, X_pure, feature_names=STUDENT_FEATURES,
                      max_display=15, show=False, title='SHAP — Pure-Thin Users')
    plt.tight_layout()
    plt.savefig('shap_pure_thin.png', dpi=120, bbox_inches='tight')
    plt.close()           # release figure memory
    del sv_pure
    gc.collect()

# Clean up subset frames
del X_near, X_pure, X_thin_val, mode_thin_val
gc.collect()


100%|===================| 443/445 [00:30<00:00]        

0

## 8. Human-Readable Explanation Templates
Maps feature names + direction to plain English sentences judges and users understand.

In [9]:
# Template: (feature_name, high_impact_text, low_impact_text)
# 'high' = SHAP > 0 (increases default risk), 'low' = SHAP < 0 (reduces default risk)
EXPLANATION_TEMPLATES = {
    'FINANCIAL_PRESSURE':  ('High combined financial burden increased risk',
                            'Low financial pressure reduced risk'),
    'STABILITY_SCORE':     ('Low employment and asset stability increased risk',
                            'Strong employment history and asset ownership reduced risk'),
    'EMI_BURDEN':          ('EMI obligations are high relative to income',
                            'EMI obligations are manageable relative to income'),
    'LOAN_TO_INCOME':      ('Loan amount is large relative to annual income',
                            'Loan amount is proportionate to income'),
    'DEBT_TO_INCOME':      ('Total debt relative to income is elevated',
                            'Debt-to-income ratio is healthy'),
    'INCOME_STABILITY':    ('Income stability is low — short or irregular employment',
                            'Stable long-term employment reduces default risk'),
    'INCOME_ADEQUACY':     ('Income may be insufficient to service loan obligations',
                            'Income comfortably covers loan obligations'),
    'ASSET_SCORE':         ('No significant assets to act as collateral',
                            'Ownership of property or vehicle reduces risk'),
    'EMPLOYED_YEARS':      ('Short employment duration increases uncertainty',
                            'Long employment duration indicates stability'),
    'AGE_YEARS':           ('Younger applicant with less financial history',
                            'Age indicates established financial maturity'),
    'INCOME_PER_PERSON':   ('Low income per family member increases financial stress',
                            'Adequate income per family member reduces stress'),
    'CHILDREN_RATIO':      ('High dependency ratio increases financial obligations',
                            'Low dependency ratio reduces financial obligations'),
    'ANNUITY_TO_INCOME':   ('Annual loan repayment is a large share of income',
                            'Annual repayment is a small share of income'),
    'FLAG_OWN_REALTY':     ('No property ownership noted',
                            'Property ownership is a positive stability signal'),
    'FLAG_OWN_CAR':        ('No vehicle ownership noted',
                            'Vehicle ownership is a minor positive signal'),
    'REGION_RATING_CLIENT':('Region has higher credit risk profile',
                            'Region has lower credit risk profile'),
}

def get_shap_explanation(shap_vals, feature_names, top_n=3):
    """Returns human-readable risk drivers and protective factors."""
    df = pd.DataFrame({'feature': feature_names, 'shap': shap_vals})

    drivers    = df[df['shap'] > 0].sort_values('shap', ascending=False).head(top_n)
    protective = df[df['shap'] < 0].sort_values('shap').head(top_n)

    def to_text(row, direction):
        feat = row['feature']
        if feat in EXPLANATION_TEMPLATES:
            return EXPLANATION_TEMPLATES[feat][0 if direction == 'high' else 1]
        # Fallback for unmapped features
        readable = feat.replace('_',' ').title()
        return f"{'High' if direction == 'high' else 'Low'} {readable}"

    return (
        [to_text(r, 'high') for _, r in drivers.iterrows()],
        [to_text(r, 'low')  for _, r in protective.iterrows()]
    )

print('Explanation templates ready.')

Explanation templates ready.


## 9. Uncertainty Scoring
Confidence = distance from decision boundary (p=0.5).
Low-confidence thin-file users get REVIEW regardless of ML output.

In [10]:
def get_confidence(prob):
    """0 = maximally uncertain (p=0.5), 1 = maximally confident (p=0 or p=1)."""
    return round(abs(prob - 0.5) * 2, 3)

def get_confidence_label(confidence):
    if confidence >= 0.70: return 'High'
    if confidence >= 0.40: return 'Medium'
    return 'Low'

# Low confidence overrides APPROVE → REVIEW
CONFIDENCE_OVERRIDE_THRESHOLD = 0.30   # if confidence < this, force REVIEW

print('Confidence scoring defined.')
print('Low confidence (< 0.30) will override APPROVE → REVIEW.')

for p in [0.03, 0.08, 0.12, 0.20, 0.48]:
    c = get_confidence(p)
    print(f'  p={p:.2f} → confidence={c:.2f} ({get_confidence_label(c)})')

Confidence scoring defined.
Low confidence (< 0.30) will override APPROVE → REVIEW.
  p=0.03 → confidence=0.94 (High)
  p=0.08 → confidence=0.84 (High)
  p=0.12 → confidence=0.76 (High)
  p=0.20 → confidence=0.60 (Medium)
  p=0.48 → confidence=0.04 (Low)


## 10. Population Percentile Ranking

In [11]:
# Compute population probability distribution (thin-file training set)
# We rank on RAW PROBABILITY, not on credit score.
# Why: prob_to_score compresses large portions of the prob range into
# scores near 900 (because most thin-file users have low default prob).
# Ranking on probability gives a meaningful, well-spread percentile.
population_probs = train_thin_preds   # raw probabilities, not scores

# Also keep scores for display
train_thin_scores = prob_to_score(train_thin_preds)

def get_population_percentile(prob, population_probs):
    """What % of thin-file applicants have a HIGHER default probability.
    i.e. how much safer is this user vs the population.
    A score of 80 means 80% of applicants are riskier than this user.
    """
    return round(float((population_probs >= prob).mean() * 100), 1)

# Save both distributions for inference
np.save('population_score_distribution.npy', train_thin_scores)
np.save('population_prob_distribution.npy',  population_probs)

print(f'Population score distribution (thin-file training set):')
print(f'  Mean  : {train_thin_scores.mean():.0f}')
print(f'  Median: {np.median(train_thin_scores):.0f}')
print(f'  P25   : {np.percentile(train_thin_scores, 25):.0f}')
print(f'  P75   : {np.percentile(train_thin_scores, 75):.0f}')
print(f'\nPopulation probability distribution:')
print(f'  Mean  : {population_probs.mean():.4f}')
print(f'  Median: {np.median(population_probs):.4f}')
print(f'  P25   : {np.percentile(population_probs, 25):.4f}')
print(f'  P75   : {np.percentile(population_probs, 75):.4f}')

# Free the large training predictions array — saved to disk above
del train_thin_preds
gc.collect()


Population score distribution (thin-file training set):
  Mean  : 849
  Median: 860
  P25   : 811
  P75   : 900

Population probability distribution:
  Mean  : 0.0793
  Median: 0.0689
  P25   : 0.0408
  P75   : 0.1077


19917

## 11. Core Scoring Function

In [12]:
population_scores = np.load('population_score_distribution.npy')
population_probs  = np.load('population_prob_distribution.npy')
with open('thresholds.json') as f:
    THRESH = json.load(f)

def score_user(user_features: pd.DataFrame, user_mode: str, user_raw: dict):
    """
    Full scoring pipeline for a single user.
    user_features : single-row DataFrame with STUDENT_FEATURES
    user_mode     : 'A_near_thin' | 'B_pure_thin'
    user_raw      : original dict (for business rule checks)
    """
    # ── 1. Model prediction ────────────────────────────────────────────────
    explainer = explainer_A if user_mode != 'B_pure_thin' else explainer_B
    model     = student_A   if user_mode != 'B_pure_thin' else student_B

    prob       = float(np.clip(model.predict(user_features)[0], 0, 1))
    score      = prob_to_score(prob)
    confidence = get_confidence(prob)

    # ── 2. ML decision ────────────────────────────────────────────────────
    ml_decision = get_ml_decision(prob, THRESH['approve'], THRESH['reject'])

    # ── 3. Confidence override ────────────────────────────────────────────
    conf_override = None
    if confidence < CONFIDENCE_OVERRIDE_THRESHOLD and ml_decision == 'APPROVE':
        ml_decision   = 'REVIEW'
        conf_override = 'Low model confidence — flagged for manual review'

    # ── 4. Business rule overlay ──────────────────────────────────────────
    final_decision, rule_triggered = apply_business_rules(
        user_raw, ml_decision, prob
    )

    # ── 5. SHAP explanation ───────────────────────────────────────────────
    sv          = explainer.shap_values(user_features)[0]
    risk_drivers, protective_factors = get_shap_explanation(sv, STUDENT_FEATURES)

    # ── 6. Population percentile (probability-based) ───────────────────────
    pct = get_population_percentile(prob, population_probs)

    return {
        'credit_score':        score,
        'tier':                get_tier(score),
        'default_probability': round(prob, 4),
        'confidence':          confidence,
        'confidence_label':    get_confidence_label(confidence),
        'ml_decision':         get_ml_decision(prob, THRESH['approve'], THRESH['reject']),
        'final_decision':      final_decision,
        'loan_eligible':       final_decision in ('APPROVE', 'REVIEW'),
        'user_mode':           user_mode,
        'population_percentile': pct,
        'population_summary':  f'Lower default risk than {pct}% of thin-file applicants',
        'risk_drivers':        risk_drivers,
        'protective_factors':  protective_factors,
        'rule_triggered':      rule_triggered,
        'confidence_note':     conf_override,
        'decision_reasoning':  _build_reasoning_trace(
            prob, score, confidence, ml_decision, final_decision,
            rule_triggered, conf_override, risk_drivers, protective_factors
        )
    }

def _build_reasoning_trace(prob, score, conf, ml_dec, final_dec,
                            rule, conf_note, drivers, protective):
    lines = []
    lines.append(f'Credit score of {score} ({get_tier(score)}) based on demographic and financial signals.')
    if protective:
        lines.append('Positive factors: ' + '; '.join(protective[:2]) + '.')
    if drivers:
        lines.append('Risk factors: ' + '; '.join(drivers[:2]) + '.')
    if conf_note:
        lines.append(conf_note + '.')
    if rule:
        lines.append(f'Business rule applied: {rule}.')
    lines.append(f'Final decision: {final_dec}.')
    return ' '.join(lines)

print('Core scoring function ready.')


Core scoring function ready.


## 12. Scenario Simulation
'What-if' analysis: how does the score change if the user improves a specific input?

In [13]:
def simulate_scenario(user_features: pd.DataFrame, user_mode: str,
                       feature_to_change: str, new_value: float,
                       user_raw: dict):
    """
    Returns delta in credit score if a single feature changes.
    Works for any STUDENT_FEATURES column.
    """
    base_result = score_user(user_features, user_mode, user_raw)

    modified = user_features.copy()
    modified[feature_to_change] = new_value

    # Recompute composite features that depend on this feature
    if feature_to_change in ('EMI_BURDEN', 'LOAN_TO_INCOME', 'CHILDREN_RATIO'):
        modified['FINANCIAL_PRESSURE'] = (
            0.40 * modified['EMI_BURDEN'].clip(0,1) +
            0.35 * modified['LOAN_TO_INCOME'].clip(0,1) +
            0.25 * modified['CHILDREN_RATIO'].clip(0,1)
        )
    if feature_to_change in ('INCOME_STABILITY', 'EMPLOYED_YEARS', 'FLAG_OWN_REALTY'):
        modified['STABILITY_SCORE'] = (
            0.50 * modified['INCOME_STABILITY'].clip(0,1) +
            0.30 * (modified['EMPLOYED_YEARS']/40).clip(0,1) +
            0.20 * modified['FLAG_OWN_REALTY'].astype(float)
        )

    modified_result = score_user(modified, user_mode, user_raw)

    delta_score    = modified_result['credit_score'] - base_result['credit_score']
    delta_prob     = modified_result['default_probability'] - base_result['default_probability']
    decision_change = (
        f"{base_result['final_decision']} → {modified_result['final_decision']}"
        if base_result['final_decision'] != modified_result['final_decision']
        else 'No change in decision'
    )

    return {
        'feature_changed':   feature_to_change,
        'old_value':         float(user_features[feature_to_change].values[0]),
        'new_value':         new_value,
        'score_before':      base_result['credit_score'],
        'score_after':       modified_result['credit_score'],
        'score_delta':       f'{delta_score:+d} points',
        'prob_delta':        f'{delta_prob:+.4f}',
        'decision_change':   decision_change,
        'interpretation':    (
            f"Reducing {feature_to_change.replace('_',' ').lower()} improves score by {delta_score} points"
            if delta_score > 0 else
            f"This change reduces score by {abs(delta_score)} points"
        )
    }

print('Scenario simulation ready.')

Scenario simulation ready.


## 13. Fairness Audit
Score distribution by income type and region. Surfacing bias is better than pretending it doesn't exist.

In [14]:
with open('label_encoders.pkl', 'rb') as f:
    encoders = pickle.load(f)

# Score the thin-file val set — use a view, not a full copy where possible
thin_val_df = X_val[thin_val.values == 1][['NAME_INCOME_TYPE', 'REGION_RATING_CLIENT']].copy()
thin_val_df['pred_prob']   = val_preds[thin_mask]
thin_val_df['credit_score'] = prob_to_score(thin_val_df['pred_prob'].values)  # vectorized
thin_val_df['true_label']   = yh_val.values[thin_mask]

# Decode income type back to labels for readability
le_income = encoders['NAME_INCOME_TYPE']
thin_val_df['income_type_label'] = le_income.inverse_transform(
    thin_val_df['NAME_INCOME_TYPE'].astype(int)
)
thin_val_df.drop(columns=['NAME_INCOME_TYPE'], inplace=True)  # free encoded col

print('=== Fairness Audit: Score Distribution by Income Type ===')
audit = thin_val_df.groupby('income_type_label').agg(
    n            = ('credit_score', 'count'),
    mean_score   = ('credit_score', 'mean'),
    median_score = ('credit_score', 'median'),
    default_rate = ('true_label',   'mean')
).round(2)
print(audit.to_string())

print('\n=== Score Distribution by Region Rating ===')
region_audit = thin_val_df.groupby('REGION_RATING_CLIENT').agg(
    n            = ('credit_score', 'count'),
    mean_score   = ('credit_score', 'mean'),
    default_rate = ('true_label',   'mean')
).round(2)
print(region_audit.to_string())

del thin_val_df, audit, region_audit
gc.collect()


=== Fairness Audit: Score Distribution by Income Type ===
                        n  mean_score  median_score  default_rate
income_type_label                                                
Businessman             1      900.00         900.0          0.00
Commercial associate  185      857.51         871.0          0.07
Pensioner              85      860.60         858.0          0.08
State servant          20      884.55         900.0          0.05
Unemployed              1      827.00         827.0          0.00
Working               155      834.86         837.0          0.08

=== Score Distribution by Region Rating ===
                        n  mean_score  default_rate
REGION_RATING_CLIENT                               
1                      77      885.10          0.01
2                     338      844.48          0.08
3                      32      844.50          0.19


11

## 14. Full Demo: End-to-End Scoring

In [15]:
# Pick a real thin-file val user
sample_idx = X_val[thin_val.values == 1].index[0]
sample_X   = X_val.loc[[sample_idx]]
sample_mode = mode_val.loc[sample_idx]
sample_raw  = sample_X.iloc[0].to_dict()  # for business rule checks

result = score_user(sample_X, sample_mode, sample_raw)

import json
print(json.dumps(result, indent=2))

{
  "credit_score": 792,
  "tier": "Excellent",
  "default_probability": 0.1277,
  "confidence": 0.745,
  "confidence_label": "High",
  "ml_decision": "REVIEW",
  "final_decision": "REVIEW",
  "loan_eligible": true,
  "user_mode": "B_pure_thin",
  "population_percentile": 17.2,
  "population_summary": "Lower default risk than 17.2% of thin-file applicants",
  "risk_drivers": [
    "High Name Education Type",
    "High Amt Goods Price",
    "High Credit To Goods"
  ],
  "protective_factors": [
    "Low Name Income Type",
    "Age indicates established financial maturity",
    "Low Amt Annuity"
  ],
  "rule_triggered": null,
  "confidence_note": null,
  "decision_reasoning": "Credit score of 792 (Excellent) based on demographic and financial signals. Positive factors: Low Name Income Type; Age indicates established financial maturity. Risk factors: High Name Education Type; High Amt Goods Price. Final decision: REVIEW."
}


In [16]:
# Scenario: what if this user's EMI burden drops from current to 0.30?
sim = simulate_scenario(
    sample_X, sample_mode,
    feature_to_change='EMI_BURDEN',
    new_value=0.30,
    user_raw=sample_raw
)
print(json.dumps(sim, indent=2))

{
  "feature_changed": "EMI_BURDEN",
  "old_value": 0.11184937861456325,
  "new_value": 0.3,
  "score_before": 792,
  "score_after": 791,
  "score_delta": "-1 points",
  "prob_delta": "+0.0012",
  "decision_change": "No change in decision",
  "interpretation": "This change reduces score by 1 points"
}


## 15. Production Inference Class

In [17]:
# Save everything needed for inference
with open('student_A.pkl', 'wb') as f: pickle.dump(student_A, f)
with open('student_B.pkl', 'wb') as f: pickle.dump(student_B, f)
with open('explainer_A.pkl', 'wb') as f: pickle.dump(explainer_A, f)
with open('explainer_B.pkl', 'wb') as f: pickle.dump(explainer_B, f)

print('All models and artifacts saved.')


class ThinFileCreditScorer:
    """
    Production inference wrapper. Load once, call .predict() per user.
    """
    def __init__(self):
        with open('student_A.pkl','rb')  as f: self.model_A    = pickle.load(f)
        with open('student_B.pkl','rb')  as f: self.model_B    = pickle.load(f)
        with open('explainer_A.pkl','rb') as f: self.exp_A     = pickle.load(f)
        with open('explainer_B.pkl','rb') as f: self.exp_B     = pickle.load(f)
        with open('label_encoders.pkl','rb') as f: self.enc    = pickle.load(f)
        with open('thresholds.json')      as f: self.thresh    = json.load(f)
        with open('feature_sets.json')    as f: self.features  = json.load(f)['student']
        self.pop_scores = np.load('population_score_distribution.npy')
        self.pop_probs  = np.load('population_prob_distribution.npy')  # NEW

    def _preprocess(self, raw: dict) -> pd.DataFrame:
        df = pd.DataFrame([raw])
        df['FLAG_OWN_CAR']    = (df.get('FLAG_OWN_CAR',    'N') == 'Y').astype(int)
        df['FLAG_OWN_REALTY'] = (df.get('FLAG_OWN_REALTY', 'N') == 'Y').astype(int)
        df['AGE_YEARS']         = -df['DAYS_BIRTH'] / 365
        df['EMPLOYED_YEARS']    = df['DAYS_EMPLOYED'].apply(lambda x: -x/365 if x < 0 else 0)
        df['DEBT_TO_INCOME']    = df['AMT_CREDIT']  / (df['AMT_INCOME_TOTAL'] + 1)
        df['ANNUITY_TO_INCOME'] = df['AMT_ANNUITY'] / (df['AMT_INCOME_TOTAL'] + 1)
        df['CREDIT_TO_GOODS']   = df['AMT_CREDIT']  / (df['AMT_GOODS_PRICE']  + 1)
        df['INCOME_PER_PERSON'] = df['AMT_INCOME_TOTAL'] / (df['CNT_FAM_MEMBERS'] + 1)
        df['CHILDREN_RATIO']    = df['CNT_CHILDREN'] / (df['CNT_FAM_MEMBERS'] + 1)
        df['EMI_BURDEN']        = df['AMT_ANNUITY'] / (df['AMT_INCOME_TOTAL'] + 1)
        df['INCOME_STABILITY']  = df['EMPLOYED_YEARS'] / (df['AGE_YEARS'] + 1)
        df['LOAN_TO_INCOME']    = df['AMT_CREDIT'] / (df['AMT_INCOME_TOTAL'] * 12 + 1)
        df['FINANCIAL_PRESSURE'] = (
            0.40 * df['EMI_BURDEN'].clip(0,1) +
            0.35 * df['LOAN_TO_INCOME'].clip(0,1) +
            0.25 * df['CHILDREN_RATIO'].clip(0,1)
        )
        df['STABILITY_SCORE'] = (
            0.50 * df['INCOME_STABILITY'].clip(0,1) +
            0.30 * (df['EMPLOYED_YEARS']/40).clip(0,1) +
            0.20 * df['FLAG_OWN_REALTY'].astype(float)
        )
        df['ASSET_SCORE'] = (
            0.60 * df['FLAG_OWN_REALTY'].astype(float) +
            0.40 * df['FLAG_OWN_CAR'].astype(float)
        )
        df['INCOME_ADEQUACY'] = (
            df['AMT_INCOME_TOTAL'] / (df['AMT_ANNUITY'] * 12 + 1)
        ).clip(0, 10)

        for col, le in self.enc.items():
            df[col] = df.get(col, 'Unknown').fillna('Unknown')
            df[col] = df[col].apply(
                lambda x: le.transform([x])[0] if x in le.classes_ else 0
            )
        for feat in self.features:
            if feat not in df.columns: df[feat] = 0
        return df[self.features].fillna(0)

    def predict(self, raw: dict) -> dict:
        X   = self._preprocess(raw)
        sig = (
            float(raw.get('bureau_count', 0) > 0) * 0.35 +
            float(raw.get('inst_count', 0) > 0)   * 0.35 +
            float(raw.get('prev_app_count', 0) > 0) * 0.20
        )
        mode = 'B_pure_thin' if sig == 0 else 'A_near_thin'
        return score_user(X, mode, raw)

    def simulate(self, raw: dict, feature: str, new_value: float) -> dict:
        """What-if: change ONE raw field, recompute everything.
        Accepts raw-space features like AMT_ANNUITY or pre-computed ones like EMI_BURDEN.
        """
        # Work in raw space first — modify the raw dict so all downstream
        # composites (FINANCIAL_PRESSURE etc.) are recomputed consistently.
        modified_raw = dict(raw)       # shallow copy — safe for scalar values
        modified_raw[feature] = new_value

        X    = self._preprocess(modified_raw)
        sig  = float(raw.get('bureau_count', 0) > 0) * 0.35
        mode = 'B_pure_thin' if sig == 0 else 'A_near_thin'
        return simulate_scenario(X, mode, feature, new_value, modified_raw)


# ── Quick test ─────────────────────────────────────────────────────────────
scorer = ThinFileCreditScorer()

test_user = {
    'CNT_CHILDREN': 1, 'CNT_FAM_MEMBERS': 3,
    'NAME_EDUCATION_TYPE': 'Higher education',
    'NAME_FAMILY_STATUS': 'Married',
    'NAME_INCOME_TYPE': 'Working',
    'OCCUPATION_TYPE': 'Laborers',
    'ORGANIZATION_TYPE': 'Business Entity Type 3',
    'NAME_HOUSING_TYPE': 'House / apartment',
    'DAYS_BIRTH': -12000, 'DAYS_EMPLOYED': -1825,
    'FLAG_OWN_CAR': 'N', 'FLAG_OWN_REALTY': 'Y',
    'AMT_INCOME_TOTAL': 180000, 'AMT_CREDIT': 450000,
    'AMT_ANNUITY': 22500, 'AMT_GOODS_PRICE': 400000,
    'REGION_POPULATION_RELATIVE': 0.035, 'REGION_RATING_CLIENT': 2,
}

print('=== Full Score Output ===')
print(json.dumps(scorer.predict(test_user), indent=2))

# Scenario: raise EMI burden — expect score to DROP (model now financially grounded)
print('\n=== Scenario: Raise EMI burden to 0.55 (high stress) ===')
print(json.dumps(scorer.simulate(test_user, 'AMT_ANNUITY', int(180000 * 0.55)), indent=2))

print('\n=== Scenario: Lower EMI burden to 0.08 (low stress) ===')
print(json.dumps(scorer.simulate(test_user, 'AMT_ANNUITY', int(180000 * 0.08)), indent=2))


All models and artifacts saved.
=== Full Score Output ===
{
  "credit_score": 863,
  "tier": "Excellent",
  "default_probability": 0.0672,
  "confidence": 0.866,
  "confidence_label": "High",
  "ml_decision": "APPROVE",
  "final_decision": "APPROVE",
  "loan_eligible": true,
  "user_mode": "B_pure_thin",
  "population_percentile": 51.4,
  "population_summary": "Lower default risk than 51.4% of thin-file applicants",
  "risk_drivers": [
    "High Amt Goods Price",
    "High Name Income Type",
    "Younger applicant with less financial history"
  ],
  "protective_factors": [
    "Low Name Education Type",
    "Long employment duration indicates stability",
    "Low Region Population Relative"
  ],
  "rule_triggered": null,
  "confidence_note": null,
  "decision_reasoning": "Credit score of 863 (Excellent) based on demographic and financial signals. Positive factors: Low Name Education Type; Long employment duration indicates stability. Risk factors: High Amt Goods Price; High Name Income